In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path
import os
import h5py
import time

from my_pkg.get_elevation_from_dsm import get_utm_3d_points_from_dsm

os.environ['PROJ_LIB'] = '/home/lty/anaconda3/envs/hloc/share/proj'

from osgeo import gdal, osr
from geopy.distance import geodesic

# from scipy.stats import pairs

from hloc import extract_features, match_features
from hloc.utils.io import get_matches, get_keypoints
from hloc.visualization import plot_images, plot_keypoints, plot_matches, read_image, add_text

from my_pkg.tools import sort_key, pixel_to_geo_coordinates, read_pairs, extract_rotation_angle, \
    draw_fusion_keyframe_traj


In [3]:
image_dir = Path("/home/lty/datasets_my/DJI/m300/")
seu_uav_dir = image_dir / "seu_uav_052411"
seu_tif_dir = image_dir / "seu_tif_m300"
output_dir = Path("/home/lty/outputs/seu0524/011")
output_dir.mkdir(exist_ok=True, parents= True)
pairs_path = output_dir/"pairs.txt"
img_list_path = output_dir/"img_list.txt"
uav_list_path = output_dir/"uav_list.txt"
tif_list_path = output_dir/"tif_list.txt"
features_path = output_dir/"features.h5"
matches_path = output_dir/"matches.h5"
loc_path = output_dir/"loc_dsm_.txt"# 保存定位结果

In [16]:
image_extensions = ['.jpg', '.jpeg', '.png', '.tif', '.tiff']
# 收集 'seu_uav' 文件夹中的所有图像文件
uav_images = [
    str(p.relative_to(image_dir))  # 获取相对于 image_dir 的路径，包含文件夹前缀
    for p in seu_uav_dir.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]

# 收集 'seu_tif' 文件夹中的所有图像文件
tif_images = [
    str(p.relative_to(image_dir))  # 获取相对于 image_dir 的路径，包含文件夹前缀
    for p in seu_tif_dir.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]
uav_images = sorted(uav_images, key = sort_key)
tif_images = sorted(tif_images, key = sort_key)
# 将 UAV 和 TIF 图像文件列表合并
img_list = uav_images + tif_images
# 打印 img_list 以验证结果
# for img in img_list:
#     print(img)
#save img_list
with open(img_list_path, 'w') as f:
    for img in img_list:
        f.write(img + "\n")

with open(uav_list_path, 'w') as f:
    for img in uav_images:
        f.write(img + "\n")
with open(tif_list_path, 'w') as f:
    for img in tif_images:
        f.write(img + "\n")

In [22]:

t0 = time.time()
extract_features.main(
    conf=extract_features.confs['superpoint_max'],
    image_dir=image_dir,
    image_list=img_list,
    feature_path = features_path,
)
t1 = time.time()
match_features.main(
    conf=match_features.confs["superpoint+lightglue"],
    pairs=pairs_path,
    features=features_path,
    matches=matches_path,
)
t2 = time.time()
print(f"Feature extraction time: {t1-t0:.3f}s")
print(f"Feature matching time: {t2-t1:.3f}s")

[2025/05/25 17:29:37 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}
[2025/05/25 17:29:37 hloc INFO] Skipping the extraction.
[2025/05/25 17:29:37 hloc INFO] Matching local features with configuration:
{'model': {'features': 'superpoint', 'name': 'lightglue'},
 'output': 'matches-superpoint-lightglue'}
100%|██████████| 31/31 [00:02<00:00, 14.87it/s]
[2025/05/25 17:29:39 hloc INFO] Finished exporting matches.


Feature extraction time: 0.193s
Feature matching time: 2.214s


In [4]:
geotransform = []
# 打开 geotransform.txt 文件
with open("/home/lty/scripts/seu_geotransform_fix.txt", "r") as f:
    # 逐行读取文件内容
    for line in f:
        # 去除行首尾的空白字符和换行符
        line = line.strip()
        if line:
            try:
                # 将字符串转换为浮点数
                value = float(line)
                # 将数值添加到列表中
                geotransform.append(value)
            except ValueError:
                print(f"无法将以下内容转换为数值：'{line}'")
                # 根据需要，可以选择跳过或停止程序
                continue
# 输出读取到的 geotransform 数据
print("Geotransform 数组：\n", geotransform)

Geotransform 数组：
 [668601.89603705, 0.03459999999999788, 0.0, 3548451.1491134795, 0.0, -0.03459999999998963]


In [7]:
fx, fy = 1543.468, 1543.468   # 焦距
cx, cy = 973.856, 533.855     # 主点

K = np.array([
    [fx,  0, cx],
    [ 0, fy, cy],
    [ 0,  0,  1]
], dtype=np.float32)
from my_pkg.get_elevation_from_dsm import get_utm_3d_points_from_dsm

In [8]:
# 创建源坐标系和目标坐标系
source_srs = osr.SpatialReference()
source_srs.ImportFromEPSG(32650)

target_srs = osr.SpatialReference()
target_srs.ImportFromEPSG(4326)

# 创建坐标转换对象
coord_transform = osr.CoordinateTransformation(source_srs, target_srs)



/home/lty/anaconda3/envs/hloc/lib/python3.8/site-packages/osgeo/osr.py:410: FutureWarning: Neither osr.UseExceptions() nor osr.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [41]:
dsm_path = "/home/lty/data/SEU/dsm.tif"
geo_utm_path = "/home/lty/paper/results/052411/elevpnp-all.txt"
geo3d_path = "/home/lty/paper/results/052411/elevpnp3d-all.txt"
dataset = gdal.Open(dsm_path)
if dataset is None:
    raise FileNotFoundError(f"无法打开 DSM 文件：{dsm_path}")

 # 获取仿射变换和影像数据
dsm_geotransform = dataset.GetGeoTransform()
band = dataset.GetRasterBand(1)
dsm_array = band.ReadAsArray()
rows, cols = dsm_array.shape
# 提取仿射变换参数
dsm_origin_x, dsm_pixel_width, _, dsm_origin_y, _, dsm_pixel_height = dsm_geotransform

pairs = read_pairs(pairs_path)
print(f"Found {len(pairs)} image pairs.")
start_x, start_y = 0, 0
x_in_map, y_in_map = 0, 0
x_origin, y_origin = 0, 0
start = time.time()
n = 0

start = time.time()
with open(loc_path, 'w') as loc_file, open(geo_utm_path, 'w') as geo_utm_file, open(geo3d_path, 'w') as geo3d_file:
    for img_uav, img_tif in pairs:
        print(f"UAV: {img_uav} - TIF: {img_tif}")
        tif_name = os.path.basename(img_tif)
        match = re.match(r"(\d+)_(\d+)_(\d+).tif", tif_name)
        if match:
            start_x = match.group(2)
            start_y = match.group(3)
            print(f"start_x: {start_x}, start_y: {start_y}")
            
        lu_geox = geotransform[0] + float(start_x) * geotransform[1]
        lu_geoy = geotransform[3] + float(start_y) * geotransform[5]

        print(f"lu_geox: {lu_geox}, lu_geoy: {lu_geoy}")
        
        kp1, kp2  = get_keypoints(features_path, img_uav), get_keypoints(features_path, img_tif)
        matches,scores = get_matches(matches_path, img_uav, img_tif)
        print(matches.shape)
        pts1 = kp1[matches[:,0]]
        pts2 = kp2[matches[:,1]]
        
        F, mask = cv2.findFundamentalMat(pts1, pts2, cv2.RANSAC, 3.0)
        
        if F is not None:
            inliers = mask.ravel().tolist()
            num_inliers = np.sum(inliers)
            print(f"Number of inliers: {num_inliers}")
             # 提取内点匹配对
            # 将 mask 转换为一维布尔数组
            inliers = mask.ravel().astype(bool)
            pts1_inliers = pts1[inliers]
            pts2_inliers = pts2[inliers]
            pts2_inliers_geo = np.zeros((len(pts2_inliers), 2))
            pts2_inliers_geo_utm = np.zeros((len(pts2_inliers), 2))
            print(pts2_inliers.shape)
            for idx, (x_pix, y_pix) in enumerate(pts2_inliers): 
                x_in_map = int(start_x) + x_pix
                y_in_map = int(start_y) + y_pix
                
                x_geo = geotransform[0] + x_in_map * geotransform[1]
                y_geo = geotransform[3] + y_in_map * geotransform[5]
                
                pts2_inliers_geo_utm[idx, 0] = x_geo
                pts2_inliers_geo_utm[idx, 1] = y_geo     
                x_geo = x_geo-lu_geox
                y_geo = y_geo-lu_geoy
            
                pts2_inliers_geo[idx, 0] = x_geo
                pts2_inliers_geo[idx, 1] = y_geo
                
            satellite_3d_utm = np.zeros((len(pts2_inliers_geo_utm), 3))
            for idx, (utm_x, utm_y) in enumerate(pts2_inliers_geo_utm):
                # print(f"utm_x: {utm_x}, utm_y: {utm_y}")
                pixel_x = int(round((utm_x - dsm_origin_x) / dsm_pixel_width))
                pixel_y = int(round((utm_y - dsm_origin_y) / dsm_pixel_height))
                # print(f"pixel_x: {pixel_x}, pixel_y: {pixel_y}")

                if 0 <= pixel_x < cols and 0 <= pixel_y < rows:
                    # noise = np.random.normal(0, 2.0)
                    elevation = dsm_array[pixel_y, pixel_x]
                else:
                    elevation = -9999  # 或者使用 np.nan
                    # print("!!!!!!!!!!!!!!!!!!!!!")
                
                satellite_3d_utm[idx] = [utm_x, utm_y, elevation]
            print(f"成功构建 3D 点集，共 {len(satellite_3d_utm)} 个点")
            z_vals = satellite_3d_utm[:, 2].reshape(-1, 1)  # (N, 1)
            satellite_3d_geo = np.hstack([pts2_inliers_geo, z_vals])  
            object_points = satellite_3d_geo.astype(np.float32)
            # (N, 2) 无人机图像上的像素坐标
            image_points = pts1_inliers.astype(np.float32)
            # solvePnP：用来估计R, t
            t1 = time.time()
            success, rvec, tvec = cv2.solvePnP(
                objectPoints=object_points,
                imagePoints=image_points,
                cameraMatrix=K,
                distCoeffs=None,  # 无畸变时设为 None
                flags=cv2.SOLVEPNP_ITERATIVE  # 可换成 EPNP/DLS/UPnP 等
            )
            t2 = time.time()
            print(f"solvePnP time: {t2-t1:.5f}s")
            if success:
                R, _ = cv2.Rodrigues(rvec)  # 将旋转向量转换为矩阵
                camera_center = -R.T @ tvec
                # 相机 z 轴方向在世界坐标系下的方向向量
                cam_forward = R @ np.array([0, 0, 1])  # shape (3,)
                print(f"Rotation matrix:\n{R}")
                # 取其在地图平面 XY 上的投影
                forward_xy = cam_forward[:2]
                
                # 计算航向角（水平旋转角），以地图 x 轴为 0°
                yaw_rad = np.arctan2(forward_xy[1], forward_xy[0])
                yaw_deg = np.degrees(yaw_rad)
                
                # 保证角度在 [0, 360) 范围
                yaw_deg = (yaw_deg + 360) % 360
                print(f"Yaw angle (degrees): {yaw_deg:.2f}")
            
                # print("位姿估计成功！")
                # print("旋转矩阵 R:\n", R)
                # print("平移向量 t:\n", tvec.ravel())
                print("相机中心（世界坐标系）:\n", camera_center.ravel())
                camera_center_geo = camera_center.ravel() + np.array([lu_geox, lu_geoy, 0])
                camerax_geo = camera_center_geo[0]
                cameray_geo = camera_center_geo[1]
                cameraz_geo = camera_center_geo[2]
                camerax_geo = float(camerax_geo)
                cameray_geo = float(cameray_geo)
                cameraz_geo = float(cameraz_geo)
                # 转换为经纬度坐标（EPSG:target_epsg）
               
                lat, lon, _ = coord_transform.TransformPoint(camerax_geo, cameray_geo, cameraz_geo)
                angle = 0
                if n==0:
                    x_origin, y_origin = camerax_geo, cameray_geo
                    # x_origin = 668906.18297640
                    # y_origin = 3548234.70643408
                    # camerax_geo = x_origin
                    # cameray_geo = y_origin
                    
                print(f"相机中心utm:：{cameray_geo}, {camerax_geo}")
                
                x_in_map  = (camerax_geo-geotransform[0])/geotransform[1]
                y_in_map  = (cameray_geo-geotransform[3])/geotransform[5]
                geo_utm_file.write(f"{camerax_geo:.8f} {cameray_geo:.8f}\n")
                geo3d_file.write(f"{camerax_geo:.8f} {cameray_geo:.8f} {cameraz_geo:.8f}\n")
                loc_file.write(f"{img_uav} {lon:.8f} {lat:.8f} {x_in_map:.8f} {y_in_map:.8f} {float(camerax_geo):.8f} {float(cameray_geo):.8f} {camerax_geo-x_origin:.10f} {y_origin-cameray_geo:.10f} {angle:.8f} {cameraz_geo}\n")
            else:
                print("solvePnP 失败，可能是点分布不均或共面")
        n+=1

end = time.time()
print(f"Time: {end-start:.3f}s")


Found 733 image pairs.
UAV: seu_uav_052411/00000.png - TIF: seu_tif_m300/8_0_1500.tif
start_x: 0, start_y: 1500
lu_geox: 668601.89603705, lu_geoy: 3548399.2491134796
(497, 2)
Number of inliers: 162
(162, 2)
成功构建 3D 点集，共 162 个点
solvePnP time: 0.00028s
Rotation matrix:
[[ 0.99979202  0.01912973 -0.00706925]
 [ 0.01375323 -0.88837984 -0.45890316]
 [-0.01505887  0.45871049 -0.88845817]]
Yaw angle (degrees): 269.12
相机中心（世界坐标系）:
 [  55.16155837 -128.71418274  119.7656499 ]
相机中心utm:：3548270.534930739, 668657.0575954193
UAV: seu_uav_052411/00001.png - TIF: seu_tif_m300/8_0_1500.tif
start_x: 0, start_y: 1500
lu_geox: 668601.89603705, lu_geoy: 3548399.2491134796
(498, 2)
Number of inliers: 140
(140, 2)
成功构建 3D 点集，共 140 个点
solvePnP time: 0.00036s
Rotation matrix:
[[ 0.99959015  0.02504002 -0.01387539]
 [ 0.01689241 -0.90723351 -0.42028801]
 [-0.02311224  0.41988137 -0.90728466]]
Yaw angle (degrees): 268.11
相机中心（世界坐标系）:
 [  56.18154563 -124.03336218  121.42825105]
相机中心utm:：3548275.215751304, 66865

In [111]:

pairs = read_pairs(pairs_path)
print(f"Found {len(pairs)} image pairs.")
start_x, start_y = 0, 0
x_in_map, y_in_map = 0, 0
x_origin, y_origin = 0, 0
start = time.time()
n = 0
with open(loc_path, 'w') as loc_file:
    for img_uav, img_tif in pairs:
        print(f"UAV: {img_uav} - TIF: {img_tif}")
        kp1, kp2  = get_keypoints(features_path, img_uav), get_keypoints(features_path, img_tif)
        matches,scores = get_matches(matches_path, img_uav, img_tif)
        print(matches.shape)
        pts1 = kp1[matches[:,0]]
        pts2 = kp2[matches[:,1]]
        
        H, _ = cv2.findHomography(pts1, pts2, cv2.RANSAC, 5.0)
        print(H)
        if H is not None:
            h_uav, w_uav = 1080, 1920
            center_uav = np.array([[w_uav / 2, h_uav / 2]], dtype=np.float32)  
            center_uav[0][0] = center_uav[0][0]# 形状为 (1, 2)
            center_uav[0][1] = center_uav[0][1]# 形状为 (1, 2)
            # 将中心点坐标转换为齐次坐标
            center_uav_homogeneous = np.array([center_uav[0][0], center_uav[0][1], 1.0])  # 形状为 (3,)
            # 通过单应性矩阵进行变换
            center_tif_homogeneous = np.dot(H, center_uav_homogeneous)  # 形状为 (3,)
            # 归一化
            center_tif = center_tif_homogeneous[:2] / center_tif_homogeneous[2]  # 形状为 (2,)
            angle = extract_rotation_angle(H)
            print(f"旋转角度 (度): {angle}")
            # 打印结果
            print(f"无人机图像中心点在tif的位置：{center_tif}")
            pixel_x, pixel_y = center_tif
            tif_name = os.path.basename(img_tif)
            match = re.match(r"(\d+)_(\d+)_(\d+).tif", tif_name)
            if match:
                start_x = match.group(2)
                start_y = match.group(3)
                x_in_map = int(start_x) + pixel_x
                y_in_map = int(start_y) + pixel_y
                print(f"无人机图像中心点在地图上的位置：{x_in_map},{y_in_map}")
            try:
                lon, lat, x_geo, y_geo = pixel_to_geo_coordinates(x_in_map, y_in_map, geotransform,source_epsg=32650)
                print(f"无人机图像中心点的经纬度：{lat}, {lon}")
                if n == 0:
                    x_origin, y_origin = x_geo, y_geo
                loc_file.write(f"{img_uav} {lon:.8f} {lat:.8f} {x_in_map:.8f} {y_in_map:.8f} {x_geo:.8f} {y_geo:.8f} {x_geo-x_origin:.10f} {y_origin-y_geo:.10f} {angle:.8f}\n")
                n+=1
            except Exception as e:
                print(f"无法计算无人机图像中心点的经纬度：{e}")
end = time.time()
print(f"Time: {end-start:.3f}s")

Found 607 image pairs.
UAV: seu_uav_052410/00000.png - TIF: seu_tif_m300/20_7500_3000.tif
(1031, 2)
[[ 2.04985412e+00  3.93952344e-01 -8.06020809e+02]
 [-1.44817152e-01  2.77946901e+00  1.25589406e+03]
 [-2.66535244e-05  2.48285050e-04  1.00000000e+00]]
旋转角度 (度): -6.365715900466643
无人机图像中心点在tif的位置：[1240.0451915  2361.58289095]
无人机图像中心点在地图上的位置：8740.045191504361,5361.582890947808
无人机图像中心点的经纬度：32.05800902662729, 118.78919163392288
UAV: seu_uav_052410/00001.png - TIF: seu_tif_m300/20_7500_3000.tif
(1030, 2)
[[ 2.04729345e+00  4.04748228e-01 -7.95858928e+02]
 [-1.51631325e-01  2.79785849e+00  1.26987841e+03]
 [-2.32604921e-05  2.49799854e-04  1.00000000e+00]]
旋转角度 (度): -6.550708609964005
无人机图像中心点在tif的位置：[1247.6671084  2368.54780706]
无人机图像中心点在地图上的位置：8747.667108399139,5368.547807056557
无人机图像中心点的经纬度：32.05800681415086, 118.78919438399845
UAV: seu_uav_052410/00002.png - TIF: seu_tif_m300/20_7500_3000.tif
(1028, 2)
[[ 1.12215125e+00 -1.46624627e-01 -7.15132857e+01]
 [-5.99843382e-01  7.84788124e-

# 生成地图轨迹

In [7]:
import cv2
from my_pkg.tools import parse_keyframe_file, draw_keyframe_trajectory, plot_traj_tif
points_traj = []
with open("/home/lty/outputs/seu0524/011/loc_dsm_3.0.txt", 'r') as loc_file:
    for line in loc_file:
        parts = line.strip().split()
        if len(parts) < 4:
            continue
        x_in_map = float(parts[3])*0.2   # 调整横坐标
        y_in_map = float(parts[4])*0.2  # 调整纵坐标
        print(f"{parts[0]}: {x_in_map}, {y_in_map}")
        points_traj.append((x_in_map, y_in_map))

    # 将点转换为整数坐标（像素坐标）
points_traj = [(int(x), int(y)) for x, y in points_traj]

# plot_traj_tif(
#     map_image_path="/home/lty/outputs/seu0524/011/gt.png",
#     loc_file_path="/home/lty/outputs/seu0524/011/loc_dsm_.txt",
#     output_image_path=output_dir/"gt_elevation_all.png",
#     scale_factor=0.2,
# )

# 绘制关键帧的单独景象匹配结果
map = cv2.imread("/home/lty/outputs/seu0524/011/gt.png", cv2.IMREAD_COLOR)
keyframe_mapping = parse_keyframe_file("/home/lty/code/ORB_SLAM3_detailed_comments/KeyFrameId.txt")
# keyframe_mapping = parse_keyframe_file("/home/lty/outputs/seu0524/011/KeyFrameId.txt")
map_with_traj = draw_keyframe_trajectory(map, points_traj, keyframe_mapping)
cv2.imwrite(output_dir/"gt_elevpnp.png", map_with_traj)

seu_uav_052411/00000.png: 318.85293855000003, 1044.012616998
seu_uav_052411/00001.png: 324.748818692, 1016.9558507260001
seu_uav_052411/00002.png: 318.892543166, 1043.134722476
seu_uav_052411/00003.png: 319.118127526, 1026.90518305
seu_uav_052411/00004.png: 311.49298130200003, 1036.1504063
seu_uav_052411/00005.png: 321.701066292, 1049.31410808
seu_uav_052411/00006.png: 328.69752153400003, 1052.92002355
seu_uav_052411/00007.png: 311.396481798, 1037.354595908
seu_uav_052411/00008.png: 312.16280866600005, 1043.612660082
seu_uav_052411/00009.png: 316.55654549400003, 1031.61241141
seu_uav_052411/00010.png: 308.41697878400004, 1035.6393914320001
seu_uav_052411/00011.png: 306.97489807000005, 1045.464700712
seu_uav_052411/00012.png: 314.715263354, 1029.7544784440001
seu_uav_052411/00013.png: 315.589800844, 1040.86308303
seu_uav_052411/00014.png: 317.09082258800004, 1039.4723376600002
seu_uav_052411/00015.png: 319.98264505400005, 1042.358285804
seu_uav_052411/00016.png: 317.04295265, 1041.61465

True

slam traj

In [8]:
#fusion traj
import cv2
from my_pkg.tools import parse_keyframe_file, draw_slam_keyframe_traj, draw_fusion_keyframe_traj
slam_traj_path = "/home/lty/paper/results/052411/proposed.txt"
def geoXY_to_pixelXY(geo_x, geo_y, geotransform):
    x = int((geo_x - geotransform[0]) / geotransform[1])
    y = int((geo_y - geotransform[3]) / geotransform[5])
    return x, y

slam_traj = []
with open(slam_traj_path, 'r') as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        Pixelx, Pixely = geoXY_to_pixelXY(float(parts[0]), float(parts[1]), geotransform)
        #print(f"{parts[0]}: {Pixelx}, {Pixely}")
        x_in_map = Pixelx*0.2  # 调整横坐标  seu need *0.2
        y_in_map = Pixely*0.2
        slam_traj.append((x_in_map, y_in_map))
        
slam_traj = [(int(x), int(y)) for x, y in slam_traj]   
map = cv2.imread(output_dir/"gt_elevpnp.png", cv2.IMREAD_COLOR)
map_with_traj = draw_fusion_keyframe_traj(map, slam_traj)
cv2.imwrite(output_dir/"gt_elevpnp_proposed.png", map_with_traj)

True

In [71]:
#slam traj
import cv2
from my_pkg.tools import parse_keyframe_file, draw_slam_keyframe_traj
slam_traj_path = "/home/lty/paper/results/052409/ORB-SLAM3.txt"
def geoXY_to_pixelXY(geo_x, geo_y, geotransform):
    x = int((geo_x - geotransform[0]) / geotransform[1])
    y = int((geo_y - geotransform[3]) / geotransform[5])
    return x, y

slam_traj = []
with open(slam_traj_path, 'r') as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        Pixelx, Pixely = geoXY_to_pixelXY(float(parts[0]), float(parts[1]), geotransform)
        #print(f"{parts[0]}: {Pixelx}, {Pixely}")
        x_in_map = Pixelx  # 调整横坐标  seu need *0.2
        y_in_map = Pixely 
        slam_traj.append((x_in_map, y_in_map))
        
slam_traj = [(int(x), int(y)) for x, y in slam_traj]   
map = cv2.imread(output_dir/"gt_.png", cv2.IMREAD_COLOR)
map_with_traj = draw_slam_keyframe_traj(map, slam_traj)
cv2.imwrite(output_dir/"compare.png", map_with_traj)

[ WARN:0@10112.892] global loadsave.cpp:241 findDecoder imread_('/home/lty/outputs/seu0524/010/gt_.png'): can't open/read file: check file path/integrity


error: OpenCV(4.10.0) /io/opencv/modules/imgcodecs/src/loadsave.cpp:798: error: (-215:Assertion failed) !_img.empty() in function 'imwrite'


In [ ]:
# 打开 HDF5 文件
with h5py.File(features_path, 'r') as f:
    # 获取所有数据集的名称
    print("文件中的数据集和分组结构：")
    def print_structure(name, obj):
        """递归打印 HDF5 文件的层次结构"""
        if isinstance(obj, h5py.Group):
            print(f"Group: {name}")
        elif isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name} - Shape: {obj.shape} - Type: {obj.dtype}")
    f.visititems(print_structure)

    # 示例：读取一个特定图像的特征
    example_image = "seu_uav/DJI_0218.JPG"  # 替换为您的图像名称
    if example_image in f:
        group = f[example_image]
        print(f"\n特定图像 '{example_image}' 的内容:")
        for key in group.keys():
            data = group[key][:]
            print(f"{key}: {data.shape} - {data.dtype}")
    else:
        print(f"图像 '{example_image}' 不存在于 HDF5 文件中。")